In [ ]:
from rdkit.Chem import Draw
from src.analysis.processing import shap_ranking, shapiq_ranking
import os
import pickle
from shapiq.interaction_values import InteractionValues
from shapiq.plot.utils import format_labels

real_dataset = "../results/real_data/data_batteries_ecfp_descriptor/explanations/"
real_models = "../results/real_data/data_batteries_ecfp_descriptor"
shap_results = 'shap_results.pickle'
shapiq_results = 'shapiq1_results.pickle'

with open(os.path.join(real_dataset, shap_results), 'rb') as f:
    shap_results = pickle.load(f)
with open(os.path.join(real_dataset, shapiq_results), 'rb') as f:
    shapiq_results = pickle.load(f)

target = 'capacity_max'
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from matplotlib.gridspec import GridSpec
from rdkit import Chem

def plot_shap(shap_values, features_names, ax):
    top_10_features = np.abs(shap_values).argsort()[-10:][::-1]
    features_names = features_names[top_10_features]
    shap_values = shap_values[top_10_features]
    sns.barplot(y=features_names, x=shap_values, orient='h', ax=ax)

def plot_shapiq(shapiq_values, features_names, ax):
    top_10_features = np.abs(shapiq_values).argsort()[-10:][::-1]
    features_names = features_names[top_10_features]
    shapiq_values = shapiq_values[top_10_features]
    sns.barplot(y=features_names, x=shapiq_values, orient='h', ax=ax)

def get_features_rf(model_path, feature_names):
    rf_model = joblib.load(model_path)
    used_features = set()
    for tree in rf_model.estimators_:
        # Get the feature indices used in the current tree's splits
        used_in_tree = [i for i in tree.tree_.feature if i != -2]
        used_features.update(used_in_tree)
    # Map indices back to feature names
    used_feature_names = [feature_names[i] for i in used_features]
    return used_feature_names

import pandas as pd

features_names = shap_results['test_data'][0].drop(columns=[target]).columns
feature_mapping = dict(enumerate(features_names))
shap_values = shap_results['shap_values']
shapiq_values = shapiq_results['interactions']
ranking_per_fold = []


feature_names_mapping = {
    'radius': 'Radius',
    'diameter': 'Diameter',
    'num_heteroatoms': '#Heteroatoms',
    'num_rotatable_bonds': '#Rotatable Bonds',
    'num_h_acceptors': '#H-bond Acceptors',
    'num_h_donors': '#H-bond Donors',
    'tpsa': 'TPSA',
    'mol_wt': 'Molecular Weight',
    'o%': 'O%',
    'n%': 'N%',
    'c%': 'C%'
}
for f_name in features_names:
    if f_name not in feature_names_mapping:
        f_n = f_name.split('_')[-1]
        feature_names_mapping[f_name] = f"ECFP {f_n}"

selected_examples = [
    (1, 5),
    (0, 0),
    (1, 2),
]

for i in range(len(shap_values)):
    fold_ranking = []
    for j, (sv, iv) in enumerate(zip(shap_values[i], shapiq_values[i])):
        if (i, j) not in selected_examples:
            continue
        smiles = shap_results['smiles'][i].iloc[j]
        print(f"Fold {i}, Example {j}, SMILES: {smiles}")
        iv = InteractionValues.from_dict(iv)
        interaction_list = iv.interaction_lookup.keys()

        feature_labels = np.array([format_labels(feature_tuple=iv, feature_mapping=feature_mapping) for iv in interaction_list])
        iv_ranking = pd.DataFrame({'features': feature_labels, 'ranking': iv.values})
        iv_ranking = iv_ranking[~iv_ranking['features'].str.contains('Base Value')]

        for f in features_names:
            if f not in feature_labels:
                new_row = pd.DataFrame({'features': [f], 'ranking': [0.]})
                iv_ranking = pd.concat([iv_ranking, new_row], ignore_index=True)

        shap_series = pd.Series(sv, index=features_names)
        shapiq_series = pd.Series(iv_ranking['ranking'].values, index=iv_ranking['features'].values)

        shap_rank = shap_series.abs().rank(method='min', ascending=False).astype(int)
        shapiq_rank = shapiq_series.abs().rank(method='min', ascending=False).astype(int)


        model_features = get_features_rf(os.path.join(real_models, f'model_{i}.joblib'), features_names)
        not_in_model = [f for f in features_names if f not in model_features]
        print(f"Features not in model {i}: {not_in_model}")

        print(f"SHAP for not in model {i}: {shap_series.loc[not_in_model]}")
        print(f"SHAP-IQ for not in model {i}: {shapiq_series.loc[not_in_model]}")

        shap_top10_features = shap_series.abs().nlargest(10).index
        shapiq_top10_features = shapiq_series.abs().nlargest(10).index
        union_features = shap_top10_features.union(shapiq_top10_features)

        shap_rank_top10 = shap_rank.loc[union_features]
        shapiq_rank_top10 = shapiq_rank.loc[union_features]
        union_features_names = [feature_names_mapping[f] for f in union_features]

        rank_df = pd.DataFrame({
            "Feature": union_features_names,
            "SHAP Rank": shap_rank_top10,
            "SHAP-IQ Rank": shapiq_rank_top10
        }).sort_values(by="SHAP Rank").reset_index(drop=True)

        agreement_df = pd.DataFrame({
            "Feature": union_features_names,
            "SHAP Value": shap_series.loc[union_features],
            "SHAP-IQ Value": shapiq_series.loc[union_features]
        }).sort_values(by="SHAP Value", key=abs, ascending=True)

        agreement_df['Sign SHAP'] = np.sign(agreement_df['SHAP Value'])
        agreement_df['Sign SHAP-IQ'] = np.sign(agreement_df['SHAP-IQ Value'])

        fig = plt.figure(layout="constrained", figsize=(15, 10))
        fig.suptitle(f'{smiles}', fontsize=20)
        gs = GridSpec(2, 3, figure=fig)
        ax1 = fig.add_subplot(gs[0, 0])
        ax2 = fig.add_subplot(gs[0, 1:])
        ax3 = gs[1, :].subgridspec(1, 2, wspace=0.1)
        ax3a = fig.add_subplot(ax3[0])

        ax3b = fig.add_subplot(ax3[1], sharex=ax3a)



        rank_df = rank_df.set_index('Feature')  # Set 'Feature' as the index for the table

        the_table = ax2.table(
            cellText=rank_df.values,
            rowLabels=rank_df.index,
            colLabels=rank_df.columns,
            loc='center',       # Center the table in the subplot
            cellLoc='center',   # Center the text in each cell

        )

        ax2.axis('off')  # Hide the axes for the table

        the_table.auto_set_font_size(False)
        the_table.set_fontsize(14)
        the_table.scale(0.8, 1.2) # Adjust height
        #
        # # Make it look more like a booktabs table
        # for key, cell in the_table.get_celld().items():
        #     cell.set_edgecolor('none') # Remove cell borders
        #     if key[0] == 0 or key[1] == -1: # Header or index
        #         cell.set_text_props(weight='bold')
        #     if key[0] == 0: # Header
        #          cell.set_height(cell.get_height() * 1.5) # More space for header

        sns.barplot(data=agreement_df, y='Feature', x='SHAP Value', orient='h', ax=ax3a, hue='Sign SHAP', palette={-1: '#1e88e5', 1: '#ff0d57'}, legend=False)
        sns.barplot(data=agreement_df, y='Feature', x='SHAP-IQ Value', orient='h', ax=ax3b, hue='Sign SHAP-IQ', palette={-1: '#1e88e5', 1: '#ff0d57'}, legend=False)
        ax3a.axvline(0, color='k', linestyle='--')
        ax3b.axvline(0, color='k', linestyle='--')
        ax3b.set_yticks([])
        ax3b.set_yticklabels([])
        ax3b.set_ylabel('')
        ax3a.tick_params(axis='both', which='major', labelsize=14)
        ax3b.tick_params(axis='both', which='major', labelsize=14)
        ax3a.set_title('SHAP Values', fontsize=16)
        ax3b.set_title('SHAP-IQ Values', fontsize=16)
        ax3a.set_ylabel('Feature', fontsize=14)
        ax3a.set_xlabel('SHAP Value', fontsize=14)
        ax3b.set_xlabel('SHAP-IQ Value', fontsize=14)
        #ax3.tight_layout()  # Adjust for suptitle

        # fig, sign_ax = plt.subplots(1, 2, figsize=(15, 6), sharey=True, sharex=True)
        # sns.barplot(data=agreement_df, y='Feature', x='SHAP Value', orient='h', ax=sign_ax[0])
        # sns.barplot(data=agreement_df, y='Feature', x='SHAP-IQ Value', orient='h', ax=sign_ax[1])
        # sign_ax[0].axvline(0, color='k', linestyle='--')
        # sign_ax[1].axvline(0, color='k', linestyle='--')
        #
        # # Color the y-axis labels: Green for agreement, Red for disagreement
        # for tick_label in sign_ax[0].get_yticklabels():
        #     feature_name = tick_label.get_text()
        #     if agreement_df.loc[agreement_df['Feature'] == feature_name, 'Sign Agreement'].iloc[0]:
        #         tick_label.set_color('green')
        #         tick_label.set_weight('bold')
        #     else:
        #         tick_label.set_color('red')
        #         tick_label.set_weight('bold')
        #plt.tight_layout() # Adjust for suptitle
        mol = Chem.MolFromSmiles(smiles)
        img = Draw.MolToImage(mol, size=(400, 400))
        ax1.imshow(img)
        ax1.axis('off')

        plt.savefig(f'../results/smiles+{i}_{j}.pdf', dpi=300, bbox_inches='tight')
        plt.show()
